# Poisson QMLE vs. Negative Binomial

## Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2

## Data

In [ ]:
df = pd.read_csv("data/polish-jvs.csv",
                 dtype={"id": np.int64, "woj": str, "public": str,
                        "size": str, "nace_division": str, "nace": str})
df["size"] = pd.Categorical(df["size"], categories=["Large", "Medium", "Small"])

## Fit both models

In [ ]:
pois = smf.glm("vacancies ~ C(size) + C(public) + C(nace)", data=df,
               family=sm.families.Poisson()).fit()

## Joint MLE for NB2 — BFGS with Poisson start values matches MASS::glm.nb
start = np.append(pois.params.values, 1.0)
nb2 = smf.negativebinomial("vacancies ~ C(size) + C(public) + C(nace)",
                           data = df).fit(start_params = start,
                                          method = "bfgs", maxiter = 500, disp = 0)
print(f"Estimated alpha (1/k): {nb2.params['alpha']:.4f}")

## Empirical comparison: robust SEs vs. NB2 SEs

In [ ]:
pois_robust = smf.glm("vacancies ~ C(size) + C(public) + C(nace)", data=df,
                      family=sm.families.Poisson()).fit(cov_type="HC0")
idx = pois.bse.index[:5]
pd.DataFrame({
    "Poisson_naive":  pois.bse[idx].values,
    "Poisson_robust": pois_robust.bse[idx].values,
    "NB2":           nb2.bse[idx].values
}, index=idx).round(4)

## Point estimates are (nearly) identical

In [ ]:
idx = pois.params.index[:5]
pd.DataFrame({
    "Poisson": pois.params[idx].values,
    "NB2":     nb2.params[idx].values,
    "Diff":    (pois.params[idx] - nb2.params[idx]).values
}, index=idx).round(5)